In [1]:
import sys
sys.path.append("./tools/")

from Matave import Matave
import json
import time

from nltk.tokenize import word_tokenize
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

In [2]:
K_RANGE = list(range(3, 20))
TOP_N = 10

In [3]:
try:
    with open('./dataProcessed/nurseNotes.json', 'r') as file:
        nurse_notes = json.load(file)
    print("File loaded successfully.")
    
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")

File loaded successfully.


In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def matave_analysis(texts):
    texts = nurse_notes[list(nurse_notes.keys())[0]]
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    print(f"Number of texts: {len(texts)}")

    # MATAVE
    start = time.time()
    matave = Matave(texts)
    matave.fit(k_range = K_RANGE)
    cluster_topics = [topic.split() for topic in matave.top_topic_words.values()]
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)

    matave.visualize()

In [6]:
all_texts = []
for key in nurse_notes.keys():
    print(f"-----------{key}-----------")
    matave_analysis(nurse_notes[key])
    all_texts.extend(nurse_notes[key])

-----------P1-----------
Number of texts: 600


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.



Coherence: 0.4966063464435724
Diversity: 0.5625
Inverse Redundancy: 0.8357142857142857
Time (seconds): 3.104175090789795
----- Cluster Topics -----
['form', 'good', 'mobilise', 'restaurant', 'mobile', 'relaxed', 'voice', 'morning', 'chart', 'med']
['toilette', 'asleep', 'comfortable', 'peacefully', 'require', 'assist', 'ab', 'abdomen', 'abs', 'abx']
['change', 'peaceful', 'asleep', 'toilette', 'nil', 'accordingly', 'regular', 'safety', 'antibiotic', 'require']
['peaceful', 'toilette', 'asleep', 'antibiotic', 'safety', 'ab', 'abdomen', 'abs', 'abx', 'accept']
['restaurant', 'mobile', 'steroid', 'meal', 'form', 'mg', 'lunch', 'usual', 'attend', 'chart']
['sleep', 'night', 'medication', 'independently', 'settle', 'voice', 'continue', 'concern', 'med', 'take']
['sleep', 'night', 'medication', 'independently', 'settle', 'voice', 'continue', 'concern', 'med', 'take']
['night', 'bed', 'concern', 'toileting', 'take', 'med', 'appear', 'safety', 'sleep', 'comfortable']


In [7]:
matave_analysis(all_texts)

Number of texts: 600


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.



Coherence: 0.4966063464435724
Diversity: 0.5625
Inverse Redundancy: 0.8357142857142857
Time (seconds): 3.077979326248169
----- Cluster Topics -----
['form', 'good', 'mobilise', 'restaurant', 'mobile', 'relaxed', 'voice', 'morning', 'chart', 'med']
['toilette', 'asleep', 'comfortable', 'peacefully', 'require', 'assist', 'ab', 'abdomen', 'abs', 'abx']
['change', 'peaceful', 'asleep', 'toilette', 'nil', 'accordingly', 'regular', 'safety', 'antibiotic', 'require']
['peaceful', 'toilette', 'asleep', 'antibiotic', 'safety', 'ab', 'abdomen', 'abs', 'abx', 'accept']
['restaurant', 'mobile', 'steroid', 'meal', 'form', 'mg', 'lunch', 'usual', 'attend', 'chart']
['sleep', 'night', 'medication', 'independently', 'settle', 'voice', 'continue', 'concern', 'med', 'take']
['sleep', 'night', 'medication', 'independently', 'settle', 'voice', 'continue', 'concern', 'med', 'take']
['night', 'bed', 'concern', 'toileting', 'take', 'med', 'appear', 'safety', 'sleep', 'comfortable']
